# SIMD 与 SIMT 混合编程：内存层级

## 概述

上一节学习了混合编程中核函数与 VF 函数的调用层级。本节学习混合编程场景下的内存层级：整体存储资源如何分层，UB（Unified Buffer）这块核内共享存储如何在静态内存、动态内存和 SIMT Data Cache 之间划分，以及不同执行单元访问 UB / GM 时各自属于哪条流水线。

### 前置要求

- 已学习 3.5.2 混合编程中的核函数与 VF 函数，理解 SIMT VF、SIMD VF 的调用方式。
- 已学习 3.4.4 SIMT 内存层级，理解 Shared Memory 和 Data Cache 的基本概念。
- 本小节为理论讲解，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 说明混合编程场景下从 Global Memory 到私有内存层的三层内存结构。
- 理解 UB 空间中静态内存、动态内存、预留空间和 SIMT Data Cache 各自的作用与大小关系。
- 说明 SIMT VF、SIMD VF、Main Scalar 访问 UB / GM 时分别属于哪条流水线，以及为什么需要关注 GM 写入后的内存一致性。

### 小节内容

- 三层内存结构
- UB 空间划分
- 数据通路与流水线归属

## 三层内存结构

了解了混合编程的调用层级后，我们再来学习内存层级。在混合编程场景下，整体内存资源自慢到快分为三层：

- **Global Memory（核外全局内存）**：所有核共享，容量最大，但访问效率最低。
- **AIC 的 L1 / AIV 的 UB**：单核内的共享内存，容量较小，但访问效率较高。
- **私有内存层**：最靠近计算单元，容量最小，访问效率最高。

![](images/03_05_hybrid/hybrid_memory_hierarchy.png)

其中，UB 是 AI Vector 核内的重要片上存储，可被 SIMD VF 和 SIMT VF 共同访问。UB 的一个典型用途是在不同 Vector Function 之间复用数据：Vector Function 切换时，UB 中的数据不会被自动清除，因此可减少对 GM 的重复访问。

## UB 空间划分

下面结合 UB 内存分配图说明 UB 空间的划分：

![](images/03_05_hybrid/hybrid_unified_buffer_layout.png)

图中展示的是 256KB UB 空间的功能划分：从低地址到高地址依次为静态内存、动态内存、预留空间和 SIMT Data Cache 空间。其中，静态内存和动态内存是用户可申请的内存空间。

| 区域 | 说明 | 大小 |
| --- | --- | --- |
| 静态内存 | 编译时确定大小，通过数组（如 `__ubuf__ char buf[1024]`）分配 | 编译期固定 |
| 动态内存 | 位于静态内存之后，大小由 `<<<...>>>` 的 `dyn_ub_size` 指定；可通过动态数组 `extern __ubuf__ char buf[]` 使用 | 运行时配置 |
| 预留空间 | 编译器和 Ascend C 预留 | 固定 8KB |
| Data Cache | SIMT 专有，用于缓存 GM 访问 | 32KB ~ 128KB |

Data Cache 的大小不是固定值，会随用户配置的静态内存和动态内存大小变化，计算公式如下：

```text
Data Cache 空间大小 = min(UB 总大小（256KB） - 静态内存 - 动态内存 - 预留空间（8KB）, 128KB)
```

因此，开发者在申请静态内存和动态内存时，无法使用全部 256KB 的 UB 空间：其中 8KB 为编译器和 Ascend C 预留空间，另需为 SIMT 访问 GM 保留 Data Cache 空间。Data Cache 至少为 32KB，若剩余空间不足会触发校验报错。

## 数据通路

了解 UB 空间的划分后，还需要进一步关注不同执行空间与 UB / GM 之间的数据访问路径。混合编程场景下，AI Vector 核支持在 SIMT VF、SIMD VF 和 Main Scalar 执行空间访问 UB / GM，同时支持独立的 UB 到 GM 的 MTE 通路。整体数据通路如下图所示：

![](images/03_05_hybrid/hybrid_data_path.png)

不同执行单元对 UB 的访问属于不同流水：SIMT VF / SIMD VF 侧的 UB 读写属于 Vector 流水（`PIPE_V`），Main Scalar 侧的 UB 读写属于 Scalar 流水（`PIPE_S`），UB 与 GM 之间的数据搬运属于 MTE 流水，其中 GM 到 UB 属于 `PIPE_MTE2`，UB 到 GM 属于 `PIPE_MTE3`。当这些访问 UB 的操作之间存在数据依赖时，需要按依赖关系进行同步。

SIMT VF 和 Main Scalar 均可读写 GM 数据，但需要关注写 GM 后的内存一致性：SIMT VF 写 GM 时，底层会确保数据立即写出到 GM，因此其他通路可读到最新数据；Main Scalar 写 GM 时，数据会先写入 Cache，底层不保证其立即刷新到 GM，其他通路可能读到旧数据。此时可通过 `asc_dcci` 进行缓存控制，将缓存数据刷新到 GM，保证后续访问读取到最新数据。

### 术语速查

<table>
  <thead>
    <tr>
      <th>术语</th>
      <th>说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>UB（Unified Buffer）</td>
      <td>AI Vector 核内的核内共享存储，可被 SIMD VF 和 SIMT VF 共同访问</td>
    </tr>
    <tr>
      <td>Data Cache</td>
      <td>UB 空间中 SIMT 专有的区域，用于缓存 GM 访问，大小随静态/动态内存占用变化，介于 32KB ~ 128KB</td>
    </tr>
    <tr>
      <td>PIPE_S / PIPE_V / PIPE_MTE2 / PIPE_MTE3</td>
      <td>分别对应 Main Scalar 访问 UB、Vector（SIMT/SIMD VF）访问 UB、GM→UB 搬运、UB→GM 搬运所属的流水线</td>
    </tr>
    <tr>
      <td>asc_dcci</td>
      <td>缓存控制接口，用于将 Main Scalar 写入 Cache 的数据主动刷新到 GM，保证内存一致性</td>
    </tr>
  </tbody>
</table>

## 小节小结

本小节学习了混合编程场景下的内存层级：

- **三层结构**：Global Memory（核外共享，容量最大、效率最低）→ AIC 的 L1 / AIV 的 UB（单核共享）→ 私有内存层（最靠近计算单元，效率最高）。
- **UB 空间划分**：256KB UB 从低地址到高地址依次是静态内存、动态内存、8KB 预留空间和 32KB~128KB 的 SIMT Data Cache，Data Cache 大小随静态/动态内存占用动态计算。
- **数据通路与流水线**：SIMT VF/SIMD VF 访问 UB 属于 `PIPE_V`，Main Scalar 访问 UB 属于 `PIPE_S`，GM↔UB 的搬运分别属于 `PIPE_MTE2`（读入）和 `PIPE_MTE3`（写出）；Main Scalar 写 GM 需要用 `asc_dcci` 主动刷新以保证一致性。

至此，SIMD 与 SIMT 混合编程模型的核心内容已学习完毕。后续的算子实践章节会以具体算子为例，展示如何运用这套模型进行开发与性能优化。

## 课后练习

本节介绍了混合编程场景下的内存层级和数据通路，请根据学习内容完成以下题目进行自测。

1. （判断题）256KB 的 UB 空间可以全部由用户自由申请为静态内存或动态内存，不存在任何预留空间。

2. （单选题）SIMT Data Cache 的大小范围是多少？  
    A. 固定 8KB  
    B. 8KB ~ 32KB  
    C. 32KB ~ 128KB  
    D. 固定 256KB  

3. （单选题）SIMT VF 和 SIMD VF 访问 UB 时属于哪条流水线？  
    A. `PIPE_S`  
    B. `PIPE_V`  
    C. `PIPE_MTE2`  
    D. `PIPE_MTE3`  

4. （多选题）以下关于混合编程内存层级的说法，哪些是正确的？  
    A. GM 到 UB 的搬运属于 `PIPE_MTE2`  
    B. UB 到 GM 的搬运属于 `PIPE_MTE3`  
    C. Main Scalar 写 GM 后其他通路一定能立即读到最新数据  
    D. SIMT VF 写 GM 时底层会确保数据立即写出  

**执行以下代码获取答案。**

In [ ]:
!cat answer/03.05.03_answer.txt
